## Business Problem

As per the business problem, I am looking to determine the safest aircraft that The Company can purchase

## The Data

The data is a csv file from the National Transport Safety Board and includes aviation accident data from 1962 to 2023 about civil aviation accidents and selected incidents in the UNited Sates and international waters.

# 1. Exploratory Data Analysis

In this section I will look at the dataset and find key features and insights.

I will first import the relevant pandas package with its alias.

## a) Import Pandas and read the csv file

In [2]:
import pandas as pd

aviation_df = pd.read_csv('Aviation_Data.csv')
aviation_df.head()

/tmp/ipykernel_73711/1615477173.py:3: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_df = pd.read_csv('Aviation_Data.csv')


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## b) EDA

Next, I will look at how many rows and columns there are.

In [40]:
aviation_df.shape

(90348, 31)

From this I can see that there are 90,348 records with 31 columns. Taking a look at the columns:

In [42]:
aviation_df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')

Since my main priority is identifying the safest aircraft for The Company I will focus on these 10 columns:
* 'Investigation.Type'
* 'Injury.Severity'
* 'Aircraft.damage'
* 'Registration.Number'
* 'Total.Fatal.Injuries'
* 'Total.Serious.Injuries'
* 'Total.Minor.Injuries'
* 'Total.Uninjured'
* 'Weather.Condition'
* 'Broad.phase.of.flight'

I will continue to explore the dataset.

In [43]:
aviation_df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [44]:
aviation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90348 entries, 0 to 90347
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      90348 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

I will move on to my main columns and examine their null values:

In [36]:
# I started by storing the main columns in a list
main_columns = ['Investigation.Type', 'Injury.Severity', 
                'Aircraft.damage', 'Registration.Number', 
                'Total.Fatal.Injuries', 'Total.Serious.Injuries', 
                'Total.Minor.Injuries', 'Total.Uninjured', 
                'Weather.Condition', 'Broad.phase.of.flight']

null_value_counts = [] # I then initialized a list that will store the corresponding count of the null values for each of the main columns

for column in main_columns:
    null_values_series = aviation_df[column].isnull().value_counts() # Get the count of null and filled values for 'column'
    shape = list(null_values_series.shape)
    if shape[0] > 1: # This essentially checks to see if the column has null values
        null_count = null_values_series.loc[True] # Get the count of the null values
        null_value_counts.append(int(null_count)) # Store the count of null values for 'column' in the list of null value counts
    else:
        null_value_counts.append(0)

null_values_dict = dict(zip(main_columns, null_value_counts)) # Create a dictionary mapping every column to its null value count
null_values_dict

{'Investigation.Type': 0,
 'Injury.Severity': 2459,
 'Aircraft.damage': 4653,
 'Registration.Number': 2841,
 'Total.Fatal.Injuries': 12860,
 'Total.Serious.Injuries': 13969,
 'Total.Minor.Injuries': 13392,
 'Total.Uninjured': 7371,
 'Weather.Condition': 5951,
 'Broad.phase.of.flight': 28624}

I will create a subset from the main dataframe that will contain the main columns only and drop the null values.

In [59]:
main_subset_df = aviation_df[main_columns].dropna()
main_subset_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 47440 entries, 0 to 63908
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Investigation.Type      47440 non-null  object 
 1   Injury.Severity         47440 non-null  object 
 2   Aircraft.damage         47440 non-null  object 
 3   Registration.Number     47440 non-null  object 
 4   Total.Fatal.Injuries    47440 non-null  float64
 5   Total.Serious.Injuries  47440 non-null  float64
 6   Total.Minor.Injuries    47440 non-null  float64
 7   Total.Uninjured         47440 non-null  float64
 8   Weather.Condition       47440 non-null  object 
 9   Broad.phase.of.flight   47440 non-null  object 
dtypes: float64(4), object(6)
memory usage: 4.0+ MB


## c) Data Analysis

Having done some EDA, I have identified my priority columns and will now work with them. I want to group my data by the registration number of the aircraft to see the accident info by aircraft.

In [64]:
investigation_table = main_subset_df.groupby('Registration.Number')['Investigation.Type'].value_counts().unstack(fill_value=0)
investigation_table = investigation_table.sort_values(by='Accident', ascending=False)
investigation_table

Investigation.Type,Accident,Incident
Registration.Number,,
NONE,298,2
N20752,7,0
N11VH,6,0
N4101E,5,0
N420SB,5,0
...,...,...
N971NA,0,1
N973VJ,0,1
N974AS,0,1


From this we can see that the aircraft with the registration number N20752 has been involved in the most accidents at 7. It is clearly unsafe and we will not recommend it to the company any time soon.

Let us now look at the aircraft that have no accidents
> **Note**: Incidents tend to be less severe than accidents so we can tolerate aircraft with a high number of incidents but **NO** accidents.

In [ ]:
zero_accident_series = investigation_table.loc[investigation_table['Accident'] == 0].sort_values(by='Incident', ascending=False)

Investigation.Type,Accident,Incident
Registration.Number,,
N31013,0,3
N803DE,0,2
N1984,0,2
N146AP,0,2
N64323,0,2
...,...,...
N6838A,0,1
N68604,0,1
N6868D,0,1
